In [ ]:
import json
import numpy as np
import pandas as pd
import time

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split

In [2]:
with open('data\sentences.json', 'r') as file:
    data = json.load(file)

In [3]:
corpus = []

for sentence in data:
    corpus.extend(sentence['text'].split(' '))           # generate the corpus by extracting text from each sentence, split the list on spaces and adding it to corpus

## Tokenize the corpus

In [4]:
tokenizer = Tokenizer(lower=False, filters='')            # this will retain the word structure like chinese-owned and will not remove hyphen
tokenizer.fit_on_texts(corpus)

In [5]:
len(tokenizer.word_index)

49346

In [6]:
# save tokenizer

tokenizer_json = tokenizer.to_json()
with open("artifacts/tokenizer_v1.json", 'w') as f:
    f.write(json.dumps(tokenizer_json))

## Preparing Input and Output combinations

In [7]:
# creating input and output combination. [1st word, second word], [1st word, 2nd word, 3rd word] and so on. last element of each list will be the output and remaining prior to last will be input

sentence_list = [item['text'] for item in data]

input_sequences = []

for sentence in sentence_list:
    tokenized_sentence = tokenizer.texts_to_sequences([sentence])[0]

    for i in range(1, len(tokenized_sentence)):
        input_sequences.append(tokenized_sentence[:i+1])


Padding

In [8]:
# padding with 0s to make all the inputs of the same length 
# padding upto 30 elements in each sequence

padded_input_sequences = pad_sequences(input_sequences, maxlen = 30, padding = 'pre')

In [9]:
padded_input_sequences

array([[    0,     0,     0, ...,     0,  9474,    87],
       [    0,     0,     0, ...,  9474,    87,    21],
       [    0,     0,     0, ...,    87,    21,  2341],
       ...,
       [    0,     0,     0, ...,   351,   302,    16],
       [    0,     0,     0, ...,   302,    16, 20129],
       [    0,     0,     0, ...,    16, 20129, 20130]],
      shape=(623489, 30), dtype=int32)

Prepare X and y

In [10]:
# X --> all elements of each array in padded_input_sequences except the last element
# y --> last element of each array in padded_input_sequences

X = padded_input_sequences[:, :-1]
y = padded_input_sequences[:, -1]

Train Test Split

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
X_train.shape

(498791, 29)

## Compiling and training the model

Training on entire data

In [13]:
vocab_size = len(tokenizer.word_index) + 1

model = Sequential()
model.add(Embedding(input_dim=vocab_size, output_dim=100))
model.add(LSTM(150, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(vocab_size, activation = 'softmax'))

In [14]:
model.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

In [15]:
es = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)       # will stop the training in case of overfitting (training loss keeps going down but validation loss start going up)

In [16]:
history = model.fit(X_train, y_train, epochs = 100, verbose = 1, callbacks=[es], validation_split = 0.1)

Epoch 1/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2935s 208ms/step - accuracy: 0.0993 - loss: 7.3267 - val_accuracy: 0.1379 - val_loss: 6.7763
Epoch 2/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2964s 209ms/step - accuracy: 0.1620 - loss: 6.2721 - val_accuracy: 0.1686 - val_loss: 6.4343
Epoch 3/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2640s 188ms/step - accuracy: 0.1939 - loss: 5.7719 - val_accuracy: 0.1846 - val_loss: 6.3343
Epoch 4/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2853s 203ms/step - accuracy: 0.2172 - loss: 5.4016 - val_accuracy: 0.1931 - val_loss: 6.3029
Epoch 5/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2887s 202ms/step - accuracy: 0.2359 - loss: 5.1173 - val_accuracy: 0.1985 - val_loss: 6.2929
Epoch 6/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2847s 203ms/step - accuracy: 0.2520 - loss: 4.8857 - val_accuracy: 0.2030 - val_loss: 6.3082
Epoch 7/100
14029/14029 ━━━━━━━━━━━━━━━━━━━━ 2694s 192ms/step - accuracy: 0.2655 - loss: 4.6984 - val_accuracy: 0.2060 - val_loss: 6.3254


In [ ]:
def test_model(text, model):
    for i in range(15):
        #tokenize
        token_text = tokenizer.texts_to_sequences([text])[0]

        #padding
        padded_token_text = pad_sequences([token_text], maxlen = 29, padding = 'pre')

        #predict
        import numpy as np
        pos = np.argmax(model.predict(padded_token_text))

        for word, index in tokenizer.word_index.items():
            if index == pos:
                text = text + ' ' + word

                print(text)
                time.sleep(2)

In [34]:
text = 'The company reported'
test_model(text, model)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step
The company reported a
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
The company reported a loss
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
The company reported a loss of
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 115ms/step
The company reported a loss of the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step
The company reported a loss of the world's
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
The company reported a loss of the world's largest
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step
The company reported a loss of the world's largest cryptocurrency
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
The company reported a loss of the world's largest cryptocurrency exchange
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step
The company reported a loss of the world's largest cryptocurrency exchange regulator
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
The company reported a loss of the world's largest cryptocurrency exchange regulator to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step
The company reported a loss of the world's largest 

In [35]:
model.save('artifacts\model_v2.keras')

In [30]:
# evaluating the model 

test_loss, test_accuracy = model.evaluate(X_test, y_test)
perplexity_score = np.exp(test_loss)
print(perplexity_score)

625/625 ━━━━━━━━━━━━━━━━━━━━ 25s 39ms/step - accuracy: 0.2689 - loss: 4.8347
125.7993511729725


Using only 100k input - output combinations to train the model

In [17]:
X = padded_input_sequences[:100000, :-1]        # will only take first 100k sequences for training
y = padded_input_sequences[:100000, -1]

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
vocab_size = len(tokenizer.word_index) + 1

model2 = Sequential()
model2.add(Embedding(input_dim=vocab_size, output_dim=100))
model2.add(LSTM(150, dropout=0.2, recurrent_dropout=0.2))
model2.add(Dense(vocab_size, activation = 'softmax'))

model2.compile(loss = 'sparse_categorical_crossentropy', optimizer = 'adam', metrics = ['accuracy'])

history2 = model2.fit(X_train, y_train, epochs = 100, verbose = 1, callbacks=[es], validation_split = 0.1)

Epoch 1/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 434s 190ms/step - accuracy: 0.0568 - loss: 7.9772 - val_accuracy: 0.0696 - val_loss: 7.7598
Epoch 2/100
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 419s 186ms/step - accuracy: 0.0775 - loss: 7.1089 - val_accuracy: 0.0811 - val_loss: 7.6695


In [37]:
text = 'interest rates this year'
test_model(text, model2)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
interest rates this year to
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 183ms/step
interest rates this year to the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
interest rates this year to the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 140ms/step
interest rates this year to the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 122ms/step
interest rates this year to the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
interest rates this year to the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
interest rates this year to the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step
interest rates this year to the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
interest rates this year to the the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 192ms/step
interest rates this year to the the the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step
interest rates this year to the the the the the the the the the the
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step
interest 

In [38]:
test_loss, test_accuracy = model2.evaluate(X_test, y_test)
perplexity_score = np.exp(test_loss)
print(perplexity_score)

625/625 ━━━━━━━━━━━━━━━━━━━━ 24s 37ms/step - accuracy: 0.0708 - loss: 7.6795
2163.5678735417437


## Hyperparameter Tuning using KerasTuner

In [20]:
from kerastuner.tuners import RandomSearch
from tensorflow.keras.optimizers import Adam

C:\Users\apaks\AppData\Local\Temp\ipykernel_8812\4234344044.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch


In [26]:
def build_model(hp):
    model = Sequential()

    # embedding
    embed_dim = hp.Int('embed_dim', min_value = 50, max_value = 300, step = 25)         # search all output dimensions b/w 50 and 300 with step 25
    model.add(Embedding(input_dim = vocab_size, output_dim= embed_dim))

    # lstm layer
    lstm_units = hp.Int('lstm_units', min_value = 32, max_value = 256, step = 32)
    model.add(LSTM(lstm_units, dropout=0.2, recurrent_dropout=0.2))

    # Dense layer
    model.add(Dense(vocab_size, activation = 'softmax'))

    # learning rate
    lr = hp.Choice('learning_rate', values = [0.01, 0.001, 0.0001])

    # compile model
    model.compile(loss = 'sparse_categorical_crossentropy', optimizer = Adam(lr), metrics = ['accuracy'])

    return model

In [27]:
tuner = RandomSearch(
    build_model,
    objective = 'val_accuracy',
    max_trials=10,
    directory = 'artifacts/keras-tuner', 
    project_name = 'model-tuning-v1'
    )

tuner.search(X_train, y_train, epochs = 10, validation_split = 0.1)

Reloading Tuner from artifacts/keras-tuner\model-tuning-v1\tuner0.json

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
250               |250               |embed_dim
192               |192               |lstm_units
0.01              |0.01              |learning_rate

Epoch 1/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 744s 326ms/step - accuracy: 0.0840 - loss: 7.9153 - val_accuracy: 0.0989 - val_loss: 7.7984
Epoch 2/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 723s 317ms/step - accuracy: 0.1226 - loss: 6.6629 - val_accuracy: 0.1174 - val_loss: 7.8045
Epoch 3/10
2250/2250 ━━━━━━━━━━━━━━━━━━━━ 639s 284ms/step - accuracy: 0.1461 - loss: 5.7965 - val_accuracy: 0.1221 - val_loss: 8.0716
Epoch 4/10


KeyboardInterrupt: 

In [23]:
tuner.get_best_hyperparameters()[0].values      # returns the values of the best hyperparameters

{'embed_dim': 250, 'lstm_units': 192, 'learning_rate': 0.01}

In [24]:
model_tuned = tuner.get_best_models(num_models = 1)[0]

NotFoundError: NewRandomAccessFile failed to Create/Open: artifacts/keras-tuner\model-tuning-v1\trial_02\build_config.json : The system cannot find the file specified.
; No such file or directory

In [ ]:
history_tuned = model_tuned.fit(X_train, y_train, batch_size = 32, epochs = 100, initial_epoch = 10, validation_split = 0.1)